# ⚡ WAI-Illustrious v17 — Colab một ô

**Dùng:** bật **GPU T4** → chạy ô duy nhất bên dưới (**Runtime → Run all**) → mở link public được in ra.

- **FP16 + Gradio share** là mặc định khuyến nghị; chỉ đổi khi thật sự cần.
- **VRAM auto:** HighVRAM từ 12 GB, Cân bằng 8–12 GB, Tiết kiệm dưới 8 GB; có thể chọn tay 4 mức. GPU only dễ hết VRAM khi Hires.
- Tự cài Forge Neo, checkpoint, tag tiếng Việt, app `/vi-mobile/` và **job nền txt2img/img2img/Upscale + file stream cho Android**.
- Ảnh giữ PNG/JPEG/WebP gốc; có thể bật **S3/Cloudflare R2** bằng Colab Secrets để tải qua signed URL. Không cấu hình thì dùng file local có resume Range.
- Model/ảnh nằm trong `/content`, **không mount Drive**, không tự xóa dữ liệu cũ.
- Chạy lại ô sẽ cập nhật code/extension, giữ model đã có và khởi động lại Forge để nạp bản mới. **Đợi job hiện tại kết thúc trước khi chạy lại.**

### App Android — mở lại trực tiếp, không cần dán link
Mở app → **Máy chủ** → **Mở Colab (chạy thủ công)** → chọn GPU và bấm *Runtime → Run all*.
Khi ô cài xong, chạm nút **📲 Mở lại Anime3** hiện ngay dưới log: APK nhận link tunnel và
kết nối Forge tự động, không cần copy/dán. Đây là deep link Android; PWA hoặc APK cũ vẫn có
thể copy link rồi quay lại app. Chẩn đoán phải có **Job nền & file stream**. App tạo `job_id`,
poll trạng thái, nhận manifest nhỏ rồi tải Blob; lỗi tải file chỉ retry download, không tạo lại ảnh.
Web txt2img và app dùng cùng engine.

**S3/R2 (tuỳ chọn):** thêm các Secrets `ANIME3_S3_BUCKET`, `ANIME3_S3_ENDPOINT_URL`, `ANIME3_S3_ACCESS_KEY_ID`, `ANIME3_S3_SECRET_ACCESS_KEY`; tuỳ chọn thêm `ANIME3_S3_REGION`, `ANIME3_S3_PREFIX`, `ANIME3_S3_SIGNED_TTL`. Bật quyền **Notebook access** cho các Secret đó trước khi chạy ô. Credential không nằm trong APK hay file cấu hình repo.

### Đường dẫn nhanh
- Web Forge: `<link public>/`
- App trên điện thoại không cần APK: `<link public>/vi-mobile/`
- Ảnh gốc: `/content/stable-diffusion-webui-forge/outputs/txt2img-images/`

Civitai token được đọc từ **Colab Secrets → `CIVITAI_TOKEN`** nếu có; repo public không cần GitHub token. Các tuỳ chọn CUDA/cổng/thư mục/smoke test nâng cao vẫn có trong `colab/install.sh`, không hiện ở đây.

> `/content` mất khi xóa runtime; tải ảnh cần giữ về trước. Tuân thủ điều khoản Colab và giấy phép model.


In [ ]:
#@title 🎨 Cài / cập nhật WAI v17 + Forge + App Android { display-mode: "form" }
MODEL = "fp16" #@param ["fp16", "fp8", "int8", "hswq"]
TUNNEL = "share" #@param ["share", "cloudflared"]
VRAM_LEVEL = "auto" #@param ["auto", "balanced", "highvram", "lowvram", "gpu-only"]
BRANCH = "arena/8b18be87-ai-anime3"

# Mặc định gọn cho một ô; tùy chọn nâng cao dùng trực tiếp colab/install.sh.
MODEL_DIR = "/content/models"
PORT = 7860
CUDA_INDEX = "auto"

import os
import re
import signal
import subprocess
from html import escape
from urllib.parse import quote

env = dict(os.environ)
# Không thêm checkbox token: đọc secret khi có, không ghi token vào source/output.
try:
    from google.colab import userdata
    token = userdata.get("CIVITAI_TOKEN")
    if token:
        env["CIVITAI_TOKEN"] = token
except Exception:
    pass

# S3/R2 tuỳ chọn: chỉ chuyển giá trị vào environment của process cài/Forge.
# Không in, không ghi vào notebook, settings.json hoặc APK.
S3_SECRETS = (
    "ANIME3_S3_BUCKET", "ANIME3_S3_ENDPOINT_URL", "ANIME3_S3_REGION",
    "ANIME3_S3_ACCESS_KEY_ID", "ANIME3_S3_SECRET_ACCESS_KEY",
    "ANIME3_S3_PREFIX", "ANIME3_S3_SIGNED_TTL",
)
try:
    from google.colab import userdata
    for name in S3_SECRETS:
        try:
            value = userdata.get(name)
            if value:
                env[name] = value
        except Exception:
            pass
except Exception:
    pass

import re
import subprocess
import urllib.error
import urllib.parse
import urllib.request
from pathlib import Path

MIN_INSTALLER_VERSION = "1.0.9"


def checked_installer_version(data):
    if not data.startswith(b"#!/usr/bin/env bash") or len(data) < 5000:
        raise ValueError("Không phải script bộ cài đầy đủ")
    match = re.search(rb'^INSTALLER_VERSION=[\'"](\d+\.\d+\.\d+)[\'"]$', data, re.M)
    if not match:
        raise ValueError("Không đọc được phiên bản bộ cài")
    version = match[1].decode("ascii")
    if tuple(map(int, version.split("."))) < tuple(map(int, MIN_INSTALLER_VERSION.split("."))):
        raise ValueError(f"Bộ cài quá cũ; cần từ {MIN_INSTALLER_VERSION}")
    return version


def download_installer(dest, branch, env):
    owner_repo = "manhlee1196-boop/ai-anime3"
    raw = f"https://raw.githubusercontent.com/{owner_repo}/{branch}/colab/install.sh"
    api = (f"https://api.github.com/repos/{owner_repo}/contents/colab/install.sh?ref="
           + urllib.parse.quote(branch, safe=""))
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    tmp = dest.with_suffix(".sh.tmp")
    for url in (raw, api):
        headers = {"Accept": "application/vnd.github.raw", "User-Agent": "wai-colab-installer/1.0.9"}
        if env.get("GITHUB_TOKEN"):
            headers["Authorization"] = f"Bearer {env['GITHUB_TOKEN']}"
        try:
            request = urllib.request.Request(url, headers=headers)
            with urllib.request.urlopen(request, timeout=30) as response:
                data = response.read()
            version = checked_installer_version(data)
            tmp.write_bytes(data)
            subprocess.run(["bash", "-n", str(tmp)], check=True, capture_output=True)
            tmp.replace(dest)
            print(f"✔ Đã lấy bộ cài {version} từ nhánh {branch}.", flush=True)
            return version
        except (OSError, ValueError, subprocess.CalledProcessError, urllib.error.URLError):
            continue
        finally:
            if tmp.exists():
                tmp.unlink()  # chỉ file tạm do ô này tạo; không đụng model/repo/ảnh
    # Mất mạng chỉ dùng cache đủ mới/hợp lệ; bản 1.0.9 cần source bảng ảnh đã tạo.
    try:
        version = checked_installer_version(dest.read_bytes())
        subprocess.run(["bash", "-n", str(dest)], check=True, capture_output=True)
    except (OSError, ValueError, subprocess.CalledProcessError):
        raise RuntimeError(
            f"Không lấy được bộ cài >= {MIN_INSTALLER_VERSION}. Kiểm tra mạng hoặc mở lại notebook mới; "
            "chưa chạy cài đặt và không xóa dữ liệu cũ."
        ) from None
    print(f"⚠ Không lấy được bản mới; dùng cache đã kiểm tra {version} trong /content.", flush=True)
    return version

dest = Path("/content/install.sh")
download_installer(dest, BRANCH, env)

args = ["bash", str(dest), "--model", MODEL, "--model-dir", MODEL_DIR,
        "--branch", BRANCH, "--port", str(PORT), "--tunnel", TUNNEL,
        "--cuda", CUDA_INDEX, "--vram-level", VRAM_LEVEL, "--restart"]

print(f"▶ Model {MODEL}; tunnel {TUNNEL}; VRAM {VRAM_LEVEL}. Giữ model/ảnh đã có, cập nhật code rồi mở Forge.", flush=True)
process = subprocess.Popen(args, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                           text=True, bufsize=1, start_new_session=True)
public_url = ""
try:
    for line in process.stdout:
        print(line, end="", flush=True)
        # install.sh in đúng một dòng "Link public: https://..." khi tunnel sẵn sàng.
        # Không lấy localhost và chỉ dùng URL HTTPS làm callback cho APK.
        match = re.search(r"Link public:\s*(https://[^\s]+)", line)
        if match:
            public_url = match.group(1).rstrip("/.,;)")
    rc = process.wait()
except KeyboardInterrupt:
    os.killpg(process.pid, signal.SIGTERM)
    process.wait()
    print("Đã dừng. Chạy lại ô để tiếp tục; model .part và ảnh cũ được giữ.")
    raise
finally:
    process.stdout.close()

if rc:
    raise RuntimeError(f"Bộ cài dừng với mã {rc}. Đọc lỗi ngay phía trên; không cần chạy thêm ô khác.")

print("\n✅ Hoàn tất. Dùng link public mới ở phía trên.")
print("▶ Web: mở link → txt2img → Generate.")
if public_url:
    # Không tự điều hướng/auto-click: người dùng chủ động chạm nút để Custom Tab
    # quay về APK. URL được percent-encode trước khi đưa vào custom scheme.
    direct_url = "anime3://connect?url=" + quote(public_url, safe="")
    web_app_url = public_url.rstrip("/") + "/vi-mobile/"
    try:
        from IPython.display import HTML, display
        display(HTML(
            '<p><a href="' + escape(direct_url, quote=True) + '" '
            'style="display:inline-block;padding:12px 16px;background:#7c3aed;color:white;'
            'border-radius:9px;font-weight:700;text-decoration:none">📲 Mở lại Anime3</a></p>'
            '<p>Chạm nút này để APK nhận link Colab và kết nối tự động, không cần copy/dán.</p>'
            '<p>PWA/trình duyệt: <a href="' + escape(web_app_url, quote=True) + '">mở Anime3 web</a>.</p>'
        ))
    except Exception:
        print("▶ APK Anime3: mở anime3://connect?url=" + quote(public_url, safe=""))
else:
    print("▶ Chưa đọc được link public trong log; chạy `!bash /content/install.sh --status` rồi dùng link in ra.")
print("▶ APK cũ/PWA: mở <link public>/vi-mobile/ hoặc dán link vào tab Máy chủ.")


### Chỉ khi cần kiểm tra/lấy lại link

```bash
!bash /content/install.sh --status
!bash /content/install.sh --only launch,verify --restart
```

Không cần chạy các ô chẩn đoán, ví dụ API hoặc app Gradio phụ cho thao tác tạo ảnh thông thường. Notebook `WAI_Illustrious_v17_vi.ipynb` vẫn giữ quy trình từng bước khi cần tùy biến sâu.
